In [ ]:

url_list = []
for i in range(1,8):
    url = f"https://www.1mg.com/categories/vitamins-nutrition/food-beverages-6?filter=true&page={i}"
    url_list.append(url)
url_list

In [ ]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd
import numpy as np
import seaborn as sns

# Initialize empty lists to store scraped data
Name = []    # List to store product names
Sizes = []   # List to store product sizes
MRPs = []    # List to store product maximum retail prices (MRPs)
Prices = []  # List to store product prices
URLs = []    # List to store product URLs

page = 0  # Initialize page number for pagination

# Loop to iterate through multiple pages of the website
while True:
    
    # Define headers for the HTTP request
    header={
        'Origin': 'https://www.1mg.com',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36'
    }
    
    # Construct the URL for the current page    

    url = "https://www.1mg.com/categories/vitamins-nutrition/food-beverages-6?filter=true&page=" + str(page)

    # Send an HTTP GET request to the URL with the defined headers
    res = requests.get(url=url, headers=header)
    
    # Check if the page is not found (404 status)
    if res.status_code == 404:
        print("Page not found. Exiting loop.")
        break
    else:
        # Parse the HTML content of the response using BeautifulSoup
        obj = soup(res.content, 'html.parser')
        
        # Find all product boxes on the current page
        Box = obj.find_all('div', {'class': 'style__product-box___liepi'})
        
        # Loop through each product box and extract relevant information
        for i in Box:
            # Extract product name
            name = i.find('div', {'class': 'style__pro-title___2QwJy'})
            if name:
                Name.append(name.text.strip())
            else:
                Name.append(None)
            
            # Extract product size
            size = i.find('div', {'class': 'style__pack-size___2JQG7'})
            if size:
                Sizes.append(size.text.strip())
            else:
                Sizes.append(None)
            
            # Extract product MRP (maximum retail price)
            mrps = i.find('span', {'class': 'style__discount-price___25Bya'})
            if mrps:
                MRPs.append(mrps.text.replace("₹", ""))
            else:
                MRPs.append(None)
            
            # Extract product price
            price = i.find('div', {'class': 'style__price-tag___cOxYc'})
            if price:
                Prices.append(price.text.replace("₹", ""))
            else:
                Prices.append(None)
            
            # Extract product URL
            mg_url = i.find('a', {'class': 'style__product-link___UB_67'})
            if mg_url:
                b = "https://www.1mg.com" + mg_url.get('href')
                URLs.append(b)
            else:
                URLs.append(None)
        # Print progress information for the current page
        print(page,"page=>",end=" ")
    if page == 305 :
        break
        
    page+=1 # Move to the next page for scraping
    
    print(len(URLs),end=" ")
    print(len(Prices),end=" ")
    print(len(Sizes),end=" ")
    print(len(MRPs),end=" ")
    print(len(Name))

In [ ]:
data={"name":Name,"size_of_bottle":Sizes,"MRPs":MRPs,"selling_price":Prices,"1mg_url":URLs} 

In [ ]:
df = pd.DataFrame(data) # Sort the DataFrame by MRPs in descending order
df  # Sort the DataFrame by product name

In [ ]:
df['selling_price'] = df['selling_price'].str.replace(r'MRP\s*', '', regex=True)
df


In [ ]:
df['selling_price'] = df['selling_price'].str.replace(r'Discounted Price:\s*', '', regex=True)
df.head(50)

In [ ]:
df['MRPs']=df['MRPs'].fillna(df['selling_price'])
df


In [ ]:
df['type of product'] = df['size_of_bottle'].str.split(' ').str[-1]

In [ ]:
df.head(50)

In [ ]:
df.info()


In [ ]:
df['MRPs'] = pd.to_numeric(df['MRPs'], errors='coerce')
df['selling_price'] = pd.to_numeric(df['selling_price'], errors='coerce')

In [ ]:
df.to_csv("C:\\Users\\MICILMEDS\\Documents\\Medi_final\\correct_data\\Category\\Food_bivrages.csv", index=False)  # Save the DataFrame to a CSV file